## Notebook summary

| Item | Details |
| --- | --- |
| Purpose | SE-ResNeXt-50 3-stage training on original published crops |
| Model | SE-ResNeXt-50 (ImageNet pretrained, linear head) |
| Input | Original published 224x224 crops -> SquarePad -> Resize(384) |
| Training | 3-stage: frozen(5) -> coarse-tune(15) -> fine-tune(10) |
| Loss | Plain CrossEntropyLoss |
| Selection | QWK only |
| Outputs | best_model.pth, last_model.pth, history.csv, metadata.json |
| Status | Clean 3-stage training (matches working baseline pattern) |


## Detailed config

### Identity

| Item | Value |
| --- | --- |
| Purpose | Train SE-ResNeXt-50 on original published crops, 3-stage |
| Workflow | Stage 1 (frozen head) -> Stage 2 (coarse-tune) -> Stage 3 (fine-tune) |
| Output dir | `/content/drive/MyDrive/Models/seresnext50_32x4d_original/<TIMESTAMP>/` |

### Dataset

| Item | Value |
| --- | --- |
| Classes | 5 KL grades (0-4) |
| Dataset root | `/content/drive/MyDrive/Datasets/KneeXrayData_Mendeley_v1/extracted/KneeXrayData/ClsKLData/kneeKL224` |
| Input size | 384x384 |
| Augmentation | CLAHE -> SquarePad -> PIL -> HFlip(p=0.5) -> Rotation(5) -> ColorJitter(0.08,0.08) -> Resize(384) -> RandomErasing(0.10) -> ImageNet norm |

### Training

| Stage | Epochs | Head LR | Backbone LR | Scope |
| --- | --- | --- | --- | --- |
| Stage 1 | 5 | 3e-4 | 0 | head only, backbone frozen |
| Stage 2 | 15 | 3e-4 | 3e-5 | head + last conv block |
| Stage 3 | 10 | 1e-5 | 1e-5 | full fine-tune |
| Total | 30 | | | |

| Item | Value |
| --- | --- |
| Batch size | 48 |
| Num workers | 2 |
| Scheduler | CosineAnnealingLR per stage (stepped once per epoch) |
| Sampler | WeightedRandomSampler (inverse-frequency, power=1.0) |
| Weight decay | 1e-4 |

### Selection

| Item | Value |
| --- | --- |
| Selection | QWK only |
| Checkpoints | best_model.pth (max selection), last_model.pth (every epoch) |


# SE-ResNeXt-50 Original Published Crops — 3-Stage Training

Train SE-ResNeXt-50 on original published 224x224 crops.
3-stage: frozen(5) -> coarse-tune(15) -> fine-tune(10).
Matches the working original-crop baseline pattern.


## 0. Setup


In [ ]:
!pip -q install 'timm>=1.0' 'h5py>=3.9'


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import json
import random
from datetime import datetime, timezone
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import average_precision_score, cohen_kappa_score, precision_recall_fscore_support
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import transforms
from tqdm.auto import tqdm


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1. Configuration


In [ ]:
# ─── Paths ───────────────────────────────────────────────────────────────────
DATASET_ROOT = Path(
    '/content/drive/MyDrive/Datasets/KneeXrayData_Mendeley_v1/'
    'extracted/KneeXrayData/ClsKLData/kneeKL224'
)

# ─── Training ─────────────────────────────────────────────────────────────────
SEED = 42
INPUT_SIZE = 224        # native 224x224 resolution
IS_A100 = torch.cuda.is_available() and "A100" in torch.cuda.get_device_name(0)
BATCH_SIZE = 32 if IS_A100 else 16
NUM_WORKERS = 8 if IS_A100 else 2
PERSISTENT_WORKERS = False   # don't reserve GPU memory via idle workers
NUM_WORKERS = 2
EPOCHS_S1, EPOCHS_S2, EPOCHS_S3 = 5, 15, 10
TOTAL_EPOCHS = EPOCHS_S1 + EPOCHS_S2 + EPOCHS_S3
WEIGHT_DECAY = 1e-4
LR_HEAD_S1 = 3e-4
LR_HEAD_S2 = 3e-4
LR_BACKBONE_S2 = 3e-5
LR_S3 = 1e-5
SAMPLER_POWER = 1.0

# ─── Derived ────────────────────────────────────────────────────────────────
RUN_TIMESTAMP = datetime.now(timezone.utc).strftime('%Y-%m-%d_%H-%M-%S_%f_UTC')
RUN_DIR = Path('/content/drive/MyDrive/Models/seresnext50_32x4d_original') / RUN_TIMESTAMP

for p in (DATASET_ROOT,):
    if not p.exists():
        raise FileNotFoundError(p)
RUN_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)
print(f'Stages: {EPOCHS_S1}/{EPOCHS_S2}/{EPOCHS_S3} epochs  Total: {TOTAL_EPOCHS}')
print(f'LR: S1={LR_HEAD_S1}  S2={LR_HEAD_S2}/{LR_BACKBONE_S2}  S3={LR_S3}')
print(f'Batch size: {BATCH_SIZE}  Workers: {NUM_WORKERS}')
print(f'Scheduler: CosineAnnealingLR per stage')


Device: cuda
Stages: 5/15/10 epochs  Total: 30
LR: S1=0.0003  S2=0.0003/3e-05  S3=1e-05
Batch size: 48  Workers: 2
Scheduler: CosineAnnealingLR per stage


## 2. Load train / val splits


In [ ]:
def load_split(root, split):
    paths, labels = [], []
    for grade in range(5):
        gdir = root / split / str(grade)
        if not gdir.exists():
            continue
        for img in sorted(gdir.glob('*.png')):
            paths.append(str(img))
            labels.append(grade)
    print(f'  Loaded {split}: {len(paths)} images')
    return paths, labels

print(f'Dataset: {DATASET_ROOT}')
train_paths, train_labels = load_split(DATASET_ROOT, 'train')
val_paths,   val_labels   = load_split(DATASET_ROOT, 'val')

class_counts = np.bincount(train_labels, minlength=5)
print(f'Class counts: {dict(enumerate(class_counts))}')


Dataset: /content/drive/MyDrive/Datasets/KneeXrayData_Mendeley_v1/extracted/KneeXrayData/ClsKLData/kneeKL224
  Loaded train: 5778 images
  Loaded val: 826 images
Class counts: {0: np.int64(2286), 1: np.int64(1046), 2: np.int64(1516), 3: np.int64(757), 4: np.int64(173)}


## 3. Preprocessing & Dataset


In [ ]:
class OpenCVCLAHE:
    def __call__(self, image_rgb):
        lab = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2LAB)
        l, a, b = cv2.split(lab)
        l = cv2.createCLAHE(clipLimit=1.25, tileGridSize=(8, 8)).apply(l)
        return cv2.cvtColor(cv2.merge((l, a, b)), cv2.COLOR_LAB2RGB)

class SquarePad:
    def __call__(self, image_rgb):
        h, w = image_rgb.shape[:2]
        side = max(h, w)
        top = (side - h) // 2
        left = (side - w) // 2
        return cv2.copyMakeBorder(
            image_rgb, top, side - h - top, left, side - w - left,
            cv2.BORDER_CONSTANT, value=(0, 0, 0))

normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

train_transform = transforms.Compose([
    OpenCVCLAHE(), SquarePad(), transforms.ToPILImage(),
    transforms.RandomHorizontalFlip(p=0.50),
    transforms.RandomRotation(5),
    transforms.ColorJitter(brightness=0.08, contrast=0.08),
    transforms.Resize((INPUT_SIZE, INPUT_SIZE)),
    transforms.ToTensor(),
    transforms.RandomErasing(p=0.10, scale=(0.02, 0.05), ratio=(0.5, 2.0), value=0),
    normalize,
])

val_transform = transforms.Compose([
    OpenCVCLAHE(), SquarePad(), transforms.ToPILImage(),
    transforms.Resize((INPUT_SIZE, INPUT_SIZE)),
    transforms.ToTensor(),
    normalize,
])

class PublishedCropDataset(Dataset):
    def __init__(self, paths, labels, transform):
        self.paths = paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        img = cv2.imread(self.paths[index])
        if img is None:
            raise IOError(f'Cannot read: {self.paths[index]}')
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        return self.transform(img), int(self.labels[index])


class SEResNeXt50Model(nn.Module):
    def __init__(self):
        super().__init__()
        # NOTE: use classification mode (NOT features_only) so the original
        # layer1..layer4 attributes are preserved on the backbone — Stage 2
        # fine-tuning needs to unfreeze the last conv block by name.
        self.backbone = timm.create_model(
            'seresnext50_32x4d', pretrained=True, num_classes=0,
        )
        channels = self.backbone.num_features
        # Drop the timm classifier; we use our own nn.Linear on global-pooled features.
        self.classifier = nn.Linear(channels, 5)

    def forward(self, images):
        # timm's num_classes=0 model already does global pooling → (B, channels)
        features = self.backbone(images)
        return self.classifier(features)

    def freeze_all(self):
        for p in self.parameters():
            p.requires_grad = False
        for p in self.classifier.parameters():
            p.requires_grad = True
        print('  [Stage 1] Frozen backbone, training classifier head only')

    def unfreeze_last_block(self):
        # timm ResNet-family models expose layer1..layer4; the "last conv block"
        # is layer4. We unfreeze that plus the classifier.
        for p in self.parameters():
            p.requires_grad = False
        for p in self.backbone.layer4.parameters():
            p.requires_grad = True
        for p in self.classifier.parameters():
            p.requires_grad = True
        print('  [Stage 2] Unfrozen last conv block (layer4), training last block + classifier')

    def unfreeze_all(self):
        for p in self.parameters():
            p.requires_grad = True
        print('  [Stage 3] Full model unfrozen, fine-tuning end-to-end')


model = SEResNeXt50Model().to(DEVICE)
total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters: {total:,}  Trainable: {trainable:,}')

weights = (1.0 / np.power(class_counts, SAMPLER_POWER))[train_labels]
sampler = WeightedRandomSampler(
    torch.as_tensor(weights, dtype=torch.double), len(weights), replacement=True)

train_dataset = PublishedCropDataset(train_paths, train_labels, train_transform)
val_dataset   = PublishedCropDataset(val_paths,   val_labels,   val_transform)
VAL_BATCH = BATCH_SIZE * 2

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, sampler=sampler,
    num_workers=NUM_WORKERS, pin_memory=True,
    persistent_workers=PERSISTENT_WORKERS)
val_loader = DataLoader(
    val_dataset, batch_size=VAL_BATCH, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
    persistent_workers=PERSISTENT_WORKERS)
print(f'Train batches: {len(train_loader)}  Val batches: {len(val_loader)}')


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


model.safetensors: reconstructing file:   0%|          |  0.00B /  111MB            

model.safetensors: downloading bytes:           |  0.00B            

Total parameters: 25,521,141  Trainable: 25,521,141
Train batches: 121  Val batches: 9


## 4. Training Loop


In [ ]:
# ─── Evaluator + 3-stage training loop (head → last-block → full) ───
criterion = nn.CrossEntropyLoss()


def evaluate_full(loader):
    """Run model on `loader`, return all metrics. Selection = QWK."""
    model.eval()
    all_labels, all_preds, all_probas = [], [], []
    total_loss, total_samples = 0.0, 0
    with torch.inference_mode():
        for images, labels in tqdm(loader, desc="Evaluating"):
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)
            logits = model(images).float()
            loss = criterion(logits, labels)
            probas = F.softmax(logits, dim=1).cpu().numpy()
            preds = logits.argmax(dim=1).cpu().numpy()
            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds)
            all_probas.extend(probas)
            total_loss += loss.item() * labels.size(0)
            total_samples += labels.size(0)

    y_true = np.asarray(all_labels)
    y_pred = np.asarray(all_preds)
    y_proba = np.asarray(all_probas)
    y_onehot = np.eye(5)[y_true]
    qwk = float(cohen_kappa_score(y_true, y_pred, weights="quadratic"))
    mae = float(mean_absolute_error(y_true, y_pred))
    off1_acc = float(np.mean(np.abs(y_true - y_pred) <= 1))
    macro_f1 = float(precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )[2])
    macro_ap = float(average_precision_score(y_onehot, y_proba, average="macro"))
    accuracy = float(np.mean(y_true == y_pred))
    return {
        "loss": total_loss / max(total_samples, 1),
        "accuracy": accuracy,
        "qwk": qwk,
        "mae": mae,
        "off1_acc": off1_acc,
        "macro_f1": macro_f1,
        "macro_ap": macro_ap,
        "selection": qwk,  # QWK-only selection (matches notebook 01 convention)
    }


scaler = torch.amp.GradScaler("cuda", enabled=DEVICE.type == "cuda")

best_selection = -float("inf")
history = []

last_checkpoint_path = RUN_DIR / "last_model.pth"
best_checkpoint_path = RUN_DIR / "best_model.pth"

# --- Stage 1: classifier head only, backbone frozen -----------------------
print(f"\n=== Stage 1: {EPOCHS_S1} epochs, head-only @ lr={LR_HEAD_S1} ===")
model.freeze_all()
optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LR_HEAD_S1, weight_decay=WEIGHT_DECAY,
)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS_S1)

for epoch in range(1, EPOCHS_S1 + 1):
    model.train()
    epoch_loss, epoch_correct, epoch_total = 0.0, 0, 0
    progress = tqdm(train_loader, desc=f"[S1] Epoch {epoch}/{EPOCHS_S1}")
    for images, labels in progress:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda", enabled=DEVICE.type == "cuda"):
            logits = model(images)
            loss = criterion(logits, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        epoch_loss += loss.item() * labels.size(0)
        epoch_correct += (logits.argmax(dim=1) == labels).sum().item()
        epoch_total += labels.size(0)
        progress.set_postfix(loss=f'{epoch_loss / epoch_total:.4f}')
    scheduler.step()
    m = evaluate_full(val_loader)
    print(f'[S1] Epoch {epoch}: loss={epoch_loss/epoch_total:.4f} qwk={m["qwk"]:.4f}')
    history.append({
        "stage": 1, "epoch": epoch,
        "lr": optimizer.param_groups[0]["lr"],
        "train_loss": epoch_loss / max(epoch_total, 1),
        "train_acc": epoch_correct / max(epoch_total, 1),
        **m,
    })
    payload = {"model_state_dict": model.state_dict(), "selection": m["selection"], "metrics": m}
    torch.save(payload, last_checkpoint_path)
    if m["selection"] > best_selection:
        best_selection = m["selection"]
        torch.save(payload, best_checkpoint_path)
        print(f'  -> New best! selection={best_selection:.4f}')


# --- Stage 2: unfreeze last conv block (layer4) + classifier ----------------
print(f"\n=== Stage 2: {EPOCHS_S2} epochs, last-block + head @ lr={LR_HEAD_S2}/{LR_BACKBONE_S2} ===")
model.unfreeze_last_block()
# Two parameter groups: head (higher lr) + backbone (lower lr)
head_params = list(model.classifier.parameters())
backbone_params = [p for p in model.backbone.layer4.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(
    [{"params": head_params, "lr": LR_HEAD_S2},
     {"params": backbone_params, "lr": LR_BACKBONE_S2}],
    weight_decay=WEIGHT_DECAY,
)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS_S2)

for epoch in range(1, EPOCHS_S2 + 1):
    model.train()
    epoch_loss, epoch_correct, epoch_total = 0.0, 0, 0
    progress = tqdm(train_loader, desc=f"[S2] Epoch {epoch}/{EPOCHS_S2}")
    for images, labels in progress:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda", enabled=DEVICE.type == "cuda"):
            logits = model(images)
            loss = criterion(logits, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        epoch_loss += loss.item() * labels.size(0)
        epoch_correct += (logits.argmax(dim=1) == labels).sum().item()
        epoch_total += labels.size(0)
        progress.set_postfix(loss=f'{epoch_loss / epoch_total:.4f}')
    scheduler.step()
    m = evaluate_full(val_loader)
    print(f'[S2] Epoch {epoch}: loss={epoch_loss/epoch_total:.4f} qwk={m["qwk"]:.4f}')
    history.append({
        "stage": 2, "epoch": epoch,
        "lr": optimizer.param_groups[0]["lr"],
        "train_loss": epoch_loss / max(epoch_total, 1),
        "train_acc": epoch_correct / max(epoch_total, 1),
        **m,
    })
    payload = {"model_state_dict": model.state_dict(), "selection": m["selection"], "metrics": m}
    torch.save(payload, last_checkpoint_path)
    if m["selection"] > best_selection:
        best_selection = m["selection"]
        torch.save(payload, best_checkpoint_path)
        print(f'  -> New best! selection={best_selection:.4f}')


# --- Stage 3: full model unfrozen, fine-tune end-to-end -------------------
print(f"\n=== Stage 3: {EPOCHS_S3} epochs, full @ lr={LR_S3} ===")
model.unfreeze_all()
optimizer = torch.optim.AdamW(
    model.parameters(), lr=LR_S3, weight_decay=WEIGHT_DECAY,
)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS_S3)

for epoch in range(1, EPOCHS_S3 + 1):
    model.train()
    epoch_loss, epoch_correct, epoch_total = 0.0, 0, 0
    progress = tqdm(train_loader, desc=f"[S3] Epoch {epoch}/{EPOCHS_S3}")
    for images, labels in progress:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda", enabled=DEVICE.type == "cuda"):
            logits = model(images)
            loss = criterion(logits, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        epoch_loss += loss.item() * labels.size(0)
        epoch_correct += (logits.argmax(dim=1) == labels).sum().item()
        epoch_total += labels.size(0)
        progress.set_postfix(loss=f'{epoch_loss / epoch_total:.4f}')
    scheduler.step()
    m = evaluate_full(val_loader)
    print(f'[S3] Epoch {epoch}: loss={epoch_loss/epoch_total:.4f} qwk={m["qwk"]:.4f}')
    history.append({
        "stage": 3, "epoch": epoch,
        "lr": optimizer.param_groups[0]["lr"],
        "train_loss": epoch_loss / max(epoch_total, 1),
        "train_acc": epoch_correct / max(epoch_total, 1),
        **m,
    })
    payload = {"model_state_dict": model.state_dict(), "selection": m["selection"], "metrics": m}
    torch.save(payload, last_checkpoint_path)
    if m["selection"] > best_selection:
        best_selection = m["selection"]
        torch.save(payload, best_checkpoint_path)
        print(f'  -> New best! selection={best_selection:.4f}')


## 5. Save History & Metadata


In [ ]:
pd.DataFrame(history).to_csv(RUN_DIR / 'history.csv', index=False)

metadata = {
    'architecture': 'seresnext50_32x4d_original',
    'loss': 'cross_entropy',
    'stages': [EPOCHS_S1, EPOCHS_S2, EPOCHS_S3],
    'lr_head': [LR_HEAD_S1, LR_HEAD_S2, LR_S3],
    'lr_backbone': [0, LR_BACKBONE_S2, LR_S3],
    'weight_decay': WEIGHT_DECAY,
    'batch_size': BATCH_SIZE,
    'num_workers': NUM_WORKERS,
    'input_size': INPUT_SIZE,
    'scheduler': 'cosine_annealing',
    'sampler_power': SAMPLER_POWER,
    'best_selection': best_selection,
    'dataset_root': str(DATASET_ROOT),
    'train_samples': len(train_paths),
    'val_samples': len(val_paths),
}
with open(RUN_DIR / 'metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print('Saved:')
for fn in ['best_model.pth', 'last_model.pth', 'history.csv', 'metadata.json']:
    print(f'  {RUN_DIR / fn}')
print(f'\nBest selection: {best_selection:.4f}')


NameError: name 'history' is not defined